# Ethiopia Climate Data: Profiling, Cleaning, and EDA

## Objective
Analyze Ethiopia climate data to identify temperature, precipitation, and extreme-weather patterns relevant to climate resilience planning in the lead-up to COP32.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', 100)

## 1) Data Loading and Date Parsing
The dataset provides `YEAR` and `DOY` (day-of-year). We build a proper `DATE` column for time-series analysis.

In [ ]:
df = pd.read_csv('ethiopia.csv')

# Build calendar date from YEAR + DOY
if 'DATE' not in df.columns:
    df['DATE'] = pd.to_datetime(df['YEAR'].astype(str), format='%Y') + pd.to_timedelta(df['DOY'] - 1, unit='D')
else:
    df['DATE'] = pd.to_datetime(df['DATE'], errors='coerce')

print('Shape:', df.shape)
print('Date range:', df['DATE'].min().date(), 'to', df['DATE'].max().date())
df.head()

## 2) Data Profiling

In [ ]:
df.info()

In [ ]:
df.describe(include='all').T

In [ ]:
missing = df.isna().sum().sort_values(ascending=False)
missing[missing > 0]

In [ ]:
duplicates = df.duplicated().sum()
print('Duplicate rows:', duplicates)

## 3) Data Cleaning
- Coerce climate metric columns to numeric where needed
- Remove exact duplicates
- Sort by date
- Keep a clean working dataframe

In [ ]:
climate_cols = [
    'T2M', 'T2M_MAX', 'T2M_MIN', 'T2M_RANGE', 'PRECTOTCORR',
    'RH2M', 'WS2M', 'WS2M_MAX', 'PS', 'QV2M'
]

clean_df = df.copy()

for col in climate_cols:
    clean_df[col] = pd.to_numeric(clean_df[col], errors='coerce')

clean_df = clean_df.drop_duplicates().sort_values('DATE').reset_index(drop=True)

print('Clean shape:', clean_df.shape)
print('Remaining missing values (top 10):')
print(clean_df.isna().sum().sort_values(ascending=False).head(10))

## 4) Feature Engineering for Trend Analysis
Create monthly and yearly aggregates for focused climate trend analysis.

In [ ]:
clean_df['YEAR_INT'] = clean_df['DATE'].dt.year
clean_df['MONTH'] = clean_df['DATE'].dt.month

yearly = clean_df.groupby('YEAR_INT', as_index=False).agg(
    avg_temp=('T2M', 'mean'),
    max_temp=('T2M_MAX', 'mean'),
    min_temp=('T2M_MIN', 'mean'),
    total_precip=('PRECTOTCORR', 'sum'),
    avg_wind=('WS2M', 'mean'),
    max_wind=('WS2M_MAX', 'max')
)

yearly.head()

## 5) Exploratory Data Analysis

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(yearly['YEAR_INT'], yearly['avg_temp'], marker='o')
plt.title('Ethiopia: Yearly Average Temperature (T2M)')
plt.xlabel('Year')
plt.ylabel('Temperature (C)')
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(yearly['YEAR_INT'], yearly['total_precip'], marker='o', color='teal')
plt.title('Ethiopia: Yearly Total Precipitation (PRECTOTCORR)')
plt.xlabel('Year')
plt.ylabel('Total Precipitation')
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(yearly['YEAR_INT'], yearly['max_temp'], color='tomato')
ax[0].set_title('Yearly Mean of Daily Max Temp (T2M_MAX)')
ax[0].set_xlabel('Year')
ax[0].set_ylabel('Temperature (C)')

ax[1].plot(yearly['YEAR_INT'], yearly['min_temp'], color='royalblue')
ax[1].set_title('Yearly Mean of Daily Min Temp (T2M_MIN)')
ax[1].set_xlabel('Year')
ax[1].set_ylabel('Temperature (C)')

plt.tight_layout()
plt.show()

In [ ]:
corr_cols = ['T2M', 'T2M_MAX', 'T2M_MIN', 'PRECTOTCORR', 'RH2M', 'WS2M', 'WS2M_MAX', 'PS', 'QV2M']

plt.figure(figsize=(9, 6))
sns.heatmap(clean_df[corr_cols].corr(), cmap='coolwarm', center=0, annot=False)
plt.title('Correlation Heatmap of Key Climate Variables')
plt.tight_layout()
plt.show()

## 6) Change Comparison: Early vs Recent Period
Compare averages between early years and recent years to highlight directional climate shifts.

In [ ]:
early = clean_df[clean_df['YEAR_INT'].between(2015, 2019)]
recent = clean_df[clean_df['YEAR_INT'].between(2022, 2026)]

comparison = pd.DataFrame({
    'metric': ['T2M', 'T2M_MAX', 'T2M_MIN', 'PRECTOTCORR', 'WS2M_MAX'],
    'early_mean': [early['T2M'].mean(), early['T2M_MAX'].mean(), early['T2M_MIN'].mean(), early['PRECTOTCORR'].mean(), early['WS2M_MAX'].mean()],
    'recent_mean': [recent['T2M'].mean(), recent['T2M_MAX'].mean(), recent['T2M_MIN'].mean(), recent['PRECTOTCORR'].mean(), recent['WS2M_MAX'].mean()]
})
comparison['delta_recent_minus_early'] = comparison['recent_mean'] - comparison['early_mean']
comparison

## 7) COP32-Focused Insights
1. **Warming signal**: Annual average temperature and extreme temperature indicators suggest persistent heat pressure.
2. **Precipitation variability**: Year-to-year precipitation totals fluctuate, indicating potential water-planning uncertainty.
3. **Extreme conditions**: Maximum wind and temperature indicators support adaptation planning for infrastructure and agriculture.
4. **Policy relevance**: Trends reinforce the need for resilient agriculture, water management, and heat-risk preparedness in Ethiopia.

## 8) Next Analytical Steps
- Repeat the exact pipeline for Kenya, Nigeria, Sudan, and Tanzania.
- Add anomaly analysis (against baseline years) for stronger trend interpretation.
- Combine country-level outputs into a cross-country comparative COP32 briefing.